# Figure 11
This notebook reproduce Fig. 11 in [Ronchi et al. 2021](https://ui.adsabs.harvard.edu/abs/2021ApJ...916..100R/abstract).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import utilities.plot_settings

In [ ]:
def calculate_selection_weights(d: np.ndarray) -> np.ndarray:
    """
    Calculate the weights to assign to every star for selection.
    Weights are evaluated as a function of distance, nearest stars are easier and more likely to be detected

    Args:
        d (np.ndarray): array of distances from the Sun [kpc].

    Returns:
        (np.ndarray): array of selection weights.
    """

    # This function has been fine tuned to match the distance distribution of
    # the neutron stars with observed proper motion.
    weights = np.exp(-0.5 * d) / d

    # Normalize the weights to their sum
    w = weights / np.sum(weights)

    return w

In [ ]:
# Read observed proper motion neutron stars .csv file.
data = pd.read_csv("../../data/observations/PSRs_prop_motion_22-05-2020.csv", header=[0,1])
data.head()

In [ ]:
# Select only stars with measure of P and Pdot and that are not in globular clusters.
data = data[~data["P0"]["[s]"].isin(['NAN'])]
data = data[~data["P1"]["[s/s]"].isin(['NAN'])]
data = data[~data["DIST_DM"]["[kpc]"].isin(['NAN'])]
data = data[~data["ASSOC"]["Unnamed: 24_level_1"].isin(['EXGAL:SMC', 'EXGAL:LMC', 'GC:47Tuc', 'GC:M3', 'GC:M5', 'GC:M13', 'GC:NGC6440', 'GC:Ter5', 'GC:NGC6441', 'GC:NGC6517', 'GC:NGC6522', 'GC:NGC6624', 'GC:M28(NGC6626)', 'GC:NGC6652', 'GC:M22(NGC6656)', 'GC:NGC6752', 'GC:NGC6760', 'GC:M15', 'GC:M30'])]

In [ ]:
# Extract parameters.
RA_o = data["RAJD"]["[deg]"].to_numpy().astype(np.float64) 
DEC_o = data["DECJD"]["[deg]"].to_numpy().astype(np.float64) 
pmRA_o = data["PMRA"]["[mas/yr]"].to_numpy().astype(np.float64) 
pmRA_o_err = data["PMRA_err"]["[mas/yr]"].to_numpy().astype(np.float64) 
pmDEC_o = data["PMDEC"]["[mas/yr]"].to_numpy().astype(np.float64) 
pmDEC_o_err = data["PMDEC_err"]["[mas/yr]"].to_numpy().astype(np.float64) 
dist = data["DIST_DM"]["[kpc]"].to_numpy().astype(np.float64) 
NS_class = data["CLASS"]["Unnamed: 13_level_1"].to_numpy()
P = data["P0"]["[s]"].to_numpy().astype(np.float64) 
Pdot = data["P1"]["[s/s]"].to_numpy().astype(np.float64) 
assoc = data["ASSOC"]["Unnamed: 24_level_1"].to_numpy()

In [ ]:
# Select only isolated non recycled neutron stars i.e. with Pdot > 1e-17.
cond = (Pdot > 1e-17) & (NS_class != "Binary PSR") & (dist < 20)
RA_o = RA_o[cond]
DEC_o = DEC_o[cond]
pmRA_o = pmRA_o[cond]
pmRA_o_err = pmRA_o_err[cond]
pmDEC_o = pmDEC_o[cond]
pmDEC_o_err = pmDEC_o_err[cond]
dist = dist[cond]

# Compute the total sky proper motion.
pmtot_o = np.sqrt(pmRA_o**2 + pmDEC_o**2)

In [ ]:
# Read the final population simulated data file.
data = pd.read_pickle("../../data/paper_results/ronchi_etal_2021/simulation_maxwell_265/final_population.pkl.gz", compression="gzip")
data.head()

RA_s = data["RA"]["[deg]"].to_numpy()
DEC_s = data["DEC"]["[deg]"].to_numpy()
pmRA_s = data["v_RA"]["[mas/yr]"].to_numpy()
pmDEC_s = data["v_DEC"]["[mas/yr]"].to_numpy()
vls_s = data["v_ls"]["[km/s]"].to_numpy()
d_s = data["d"]["[kpc]"].to_numpy()

# Compute the total sky proper motion.
pmtot_s = np.sqrt(pmRA_s**2 + pmDEC_s**2)

In [ ]:
# Calculate selection weights that are function of the distance.
w = calculate_selection_weights(d_s)

In [ ]:
# Compare the distribution of the observed and simulated resampled population with K-S test and average statistics over n_trials.
n_stars = len(RA_o)
n_trials = 1000
p1_values = np.zeros(n_trials)
p2_values = np.zeros(n_trials)

n_bins = 30
d_edges = np.linspace(0, 20, n_bins + 1)
pm_edges = np.linspace(0, 300, n_bins + 1)

d_histo = np.zeros((n_trials, n_bins))
pm_histo = np.zeros((n_trials, n_bins))

# compute the bin center values
d_bin_centers = 0.5*(d_edges[1:] + d_edges[:-1])
pm_bin_centers = 0.5*(pm_edges[1:] + pm_edges[:-1])        

for i in range(n_trials):
    #print(i)
    df_select = data.sample(n_stars, replace=False, weights=w)
    d_sel = df_select["d"]["[kpc]"].to_numpy()
    pmRA_sel = df_select["v_RA"]["[mas/yr]"].to_numpy()
    pmDEC_sel = df_select["v_DEC"]["[mas/yr]"].to_numpy()
    pmtot_sel = np.sqrt(pmRA_sel**2 + pmDEC_sel**2)
                
    # save the histograms
    d_histo[i,:], _, = np.histogram(d_sel, bins=d_edges)
    pm_histo[i,:], _, = np.histogram(pmtot_sel, bins=pm_edges)
    
    # perform the K-S test on the observed and simulated samples
    stat1, p1 = stats.ks_2samp(dist, d_sel)
    stat2, p2 = stats.ks_2samp(pmtot_o, pmtot_sel)
    p1_values[i] = p1
    p2_values[i] = p2
    
print(np.mean(p1_values))
print(np.mean(p2_values))

In [ ]:
d_histo_mean = np.zeros(n_bins)   
pm_histo_mean = np.zeros(n_bins)   
d_histo_dispersion = np.zeros(n_bins)   
pm_histo_dispersion = np.zeros(n_bins)    

In [ ]:
# compute the mean and the standard deviation for the histograms produced for each resampled set
for i in range(n_bins):
    d_histo_mean[i] = np.mean(d_histo[:,i])
    pm_histo_mean[i] = np.mean(pm_histo[:,i])
    d_histo_dispersion[i] = np.std(d_histo[:,i])
    pm_histo_dispersion[i] = np.std(pm_histo[:,i])

In [ ]:
# Plot the histograms of the distance from the sun
fig, ax = plt.subplots(figsize=(8,7))
ax.hist(
    dist,
    bins=d_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1.,
    label=r"Observed",
    rasterized=True
)
ax.hist(
    d_s,
    bins=d_edges,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1.,
    label=r"Simulated all",
    rasterized=True
)
ax.hist(
    d_sel,
    bins=d_edges,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1.,
    label=r"Simulated+bias",
    zorder=5,
    rasterized=True
)
'''
ax.hist(
    d_bin_centers,
    bins=d_edges,
    weights=d_histo_mean,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1.,
    label=r"Simulated+bias",
    zorder=5,
    rasterized=True
)
ax.errorbar(
    d_bin_centers,
    d_histo_mean,
    d_histo_dispersion,
    marker=None, 
    drawstyle='steps-mid', 
    ecolor='tab:blue',
    fmt='None',
    elinewidth=2,
    alpha=1,
    rasterized=True
)
'''
ax.set_xlabel(r'$d_{\odot}$ [kpc]')
ax.set_ylabel('Number of NSs')
ax.set_xlim(0., 20.)
ax.set_yscale('log')
ax.legend(frameon=False, loc=10, fontsize=24)

plt.savefig(
    f"plots/Figure11_left.pdf",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

In [ ]:
# Total proper motion histogram plot.

fig, ax = plt.subplots(figsize=(8,7))
ax.hist(
    pmtot_o,
    bins=pm_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1.,
    label=r"Observed",
    rasterized=True
)
ax.hist(
    pmtot_s,
    bins=pm_edges,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1.,
    label=r"Simulated all",
    rasterized=True
)
ax.hist(
    pmtot_sel,
    bins=pm_edges,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1.,
    label=r"Simulated+bias",
    zorder=5,
    rasterized=True
)
'''
ax.hist(
    pm_bin_centers,
    bins=pm_edges,
    weights=pm_histo_mean,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1.,
    label=r"Simulated+bias",
    zorder=5,
    rasterized=True
)
ax.errorbar(
    pm_bin_centers,
    pm_histo_mean,
    pm_histo_dispersion,
    marker=None, 
    drawstyle='steps-mid', 
    ecolor='tab:blue',
    fmt='None',
    elinewidth=2,
    alpha=1,
    rasterized=True
)
'''
ax.set_xlabel(r"$\mu_{\rm tot}$ [mas yr$^{-1}$]")
ax.set_ylabel(r"Number of NSs")
ax.set_xlim(0., 315.)
ax.set_yscale('log')
ax.legend(frameon=False, loc=1, fontsize=24)

plt.savefig(
    f"plots/Figure11_right.pdf",
    dpi=300,
    bbox_inches="tight",
)
plt.show()